In [1]:
#کد اولیه شمارش افراد با دوربین گوشی

import cv2

# خواندن خط ب خط کد
with open("./data/object_detection_classes_coco.txt", "r") as f:
    class_names = f.read().split("\n")
# dnn = deep neural network (شبکه عصبی عمیق)
model = cv2.dnn.readNetFromTensorflow(
    "./data/frozen_inference_graph.pb",  # مدل
    "./data/ssd_mobilenet_v2_coco_2018_03_29.pbtxt"  # وزن ها
)
video = cv2.VideoCapture("http://192.168.1.100:8080/video")
# یک فریم میگیریم و ارتفاع و عرض آن را استخراج می کنیم
_, frame = video.read()
height, width, _ = frame.shape

while 1:
    ret, frame = video.read()
    # اگه فریمی نبود از ویدیو خارج شو
    if not ret:
        break

    # blob : Binary Large Object
    #blob یک نوع فریم هست
    # size = اندازه ورودی
    # swapRB = True ( تبدیل RGB به BGR )
    # crop = برش دادن عکس
    blob = cv2.dnn.blobFromImage(frame, size=(300, 300), swapRB=True, crop=False)
    model.setInput(blob)

    output = model.forward()

    number = 0  # شمارش اعداد
    for detection in output[0, 0, :, :]:
        # میزان اطمینان تشخیص
        score = float(detection[2])
        if score > 0.4:

            class_id = detection[1]  # شناسه کلاس تشخیص
            class_name = class_names[int(class_id) - 1]
            # تشخیص انسان ها
            if class_name == 'person':
                number = number + 1

                # باکس دور اشیا
                left = detection[3] * width
                top = detection[4] * height
                right = detection[5] * width
                bottom = detection[6] * height
                cv2.rectangle(frame, (int(left), int(top)), (int(right), int(bottom)),
                              (125, 0, 200), 2)
                cv2.putText(frame, class_name, (int(left), int(top) - 7),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.4, (125, 0, 200), 1)
    #نمایش تعداد کل افراد شناسایی شده
    cv2.putText(frame, f"the number of people = {number}", (20, 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 2)
    cv2.imshow("video", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):  #کلید خروج
        break

video.release()
cv2.destroyAllWindows()

In [3]:
# شمارش افراد یکبار با دروبین webcam
import cv2
import numpy as np
import time
from scipy.spatial import distance as dist
from collections import OrderedDict

# =========================
# Centroid Tracker
# =========================
class CentroidTracker:
    def __init__(self, maxDisappeared=40):
        self.nextObjectID = 0
        self.objects = OrderedDict()
        self.disappeared = OrderedDict()
        self.maxDisappeared = maxDisappeared

    def register(self, centroid):
        self.objects[self.nextObjectID] = centroid
        self.disappeared[self.nextObjectID] = 0
        self.nextObjectID += 1

    def deregister(self, objectID):
        del self.objects[objectID]
        del self.disappeared[objectID]

    def update(self, rects):
        if len(rects) == 0:
            for objectID in list(self.disappeared.keys()):
                self.disappeared[objectID] += 1

                if self.disappeared[objectID] > self.maxDisappeared:
                    self.deregister(objectID)

            return self.objects

        inputCentroids = np.zeros((len(rects), 2), dtype="int")

        for (i, (startX, startY, endX, endY)) in enumerate(rects):
            cX = int((startX + endX) / 2.0)
            cY = int((startY + endY) / 2.0)
            inputCentroids[i] = (cX, cY)

        if len(self.objects) == 0:
            for i in range(len(inputCentroids)):
                self.register(inputCentroids[i])

        else:
            objectIDs = list(self.objects.keys())
            objectCentroids = list(self.objects.values())

            D = dist.cdist(np.array(objectCentroids),
                           inputCentroids)

            rows = D.min(axis=1).argsort()
            cols = D.argmin(axis=1)[rows]

            usedRows = set()
            usedCols = set()

            for (row, col) in zip(rows, cols):

                if row in usedRows or col in usedCols:
                    continue

                objectID = objectIDs[row]
                self.objects[objectID] = inputCentroids[col]
                self.disappeared[objectID] = 0

                usedRows.add(row)
                usedCols.add(col)

            unusedRows = set(range(0, D.shape[0])).difference(usedRows)
            unusedCols = set(range(0, D.shape[1])).difference(usedCols)

            if D.shape[0] >= D.shape[1]:
                for row in unusedRows:
                    objectID = objectIDs[row]
                    self.disappeared[objectID] += 1

                    if self.disappeared[objectID] > self.maxDisappeared:
                        self.deregister(objectID)
            else:
                for col in unusedCols:
                    self.register(inputCentroids[col])

        return self.objects


# =========================
# بارگذاری مدل
# =========================
with open("./data/object_detection_classes_coco.txt", "r") as f:
    class_names = f.read().split("\n")

model = cv2.dnn.readNetFromTensorflow(
    "./data/frozen_inference_graph.pb",
    "./data/ssd_mobilenet_v2_coco_2018_03_29.pbtxt"
)

video = cv2.VideoCapture("http://10.97.21.141:8080/video")

ret, frame = video.read()
height, width, _ = frame.shape

tracker = CentroidTracker(maxDisappeared=100)

# افرادی که در 4 ساعت اخیر شمارش شده‌اند
counted_people = {}
FOUR_HOURS = 4 * 60 * 60

total_people = 0

while True:
    ret, frame = video.read()
    if not ret:
        break

    current_time = time.time()

    # حذف شناسه‌های قدیمی‌تر از 4 ساعت
    for pid in list(counted_people.keys()):
        if current_time - counted_people[pid] > FOUR_HOURS:
            del counted_people[pid]

    blob = cv2.dnn.blobFromImage(
        frame,
        size=(300, 300),
        swapRB=True,
        crop=False
    )

    model.setInput(blob)
    output = model.forward()

    rects = []

    for detection in output[0, 0, :, :]:
        score = float(detection[2])

        if score > 0.6:
            class_id = int(detection[1])
            class_name = class_names[class_id - 1]

            if class_name == "person":
                left = int(detection[3] * width)
                top = int(detection[4] * height)
                right = int(detection[5] * width)
                bottom = int(detection[6] * height)

                rects.append((left, top, right, bottom))

                cv2.rectangle(
                    frame,
                    (left, top),
                    (right, bottom),
                    (125, 0, 200),
                    2
                )

    objects = tracker.update(rects)

    for objectID, centroid in objects.items():

        if objectID not in counted_people:
            counted_people[objectID] = current_time
            total_people += 1

        text = f"ID {objectID}"
        cv2.putText(
            frame,
            text,
            (centroid[0] - 10, centroid[1] - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0, 255, 0),
            2
        )

        cv2.circle(
            frame,
            (centroid[0], centroid[1]),
            4,
            (0, 255, 0),
            -1
        )

    cv2.putText(
        frame,
        f"Total unique people: {total_people}",
        (20, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 0, 255),
        2
    )

    cv2.imshow("Video", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

video.release()
cv2.destroyAllWindows()



In [ ]:
import cv2
import face_recognition
import time
import numpy as np

video = cv2.VideoCapture("http://192.168.1.100:8080/video")

known_faces = []
last_seen = []
total_people = 0

FOUR_HOURS = 4 * 60 * 60

while True:
    ret, frame = video.read()
    if not ret:
        break

    current_time = time.time()

    # حذف افراد قدیمی‌تر از 4 ساعت
    i = 0
    while i < len(last_seen):
        if current_time - last_seen[i] > FOUR_HOURS:
            del last_seen[i]
            del known_faces[i]
        else:
            i += 1

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    face_locations = face_recognition.face_locations(rgb)
    face_encodings = face_recognition.face_encodings(
        rgb,
        face_locations
    )

    for face_encoding in face_encodings:

        matches = face_recognition.compare_faces(
            known_faces,
            face_encoding,
            tolerance=0.5
        )

        if True not in matches:
            known_faces.append(face_encoding)
            last_seen.append(current_time)
            total_people += 1
        else:
            idx = matches.index(True)
            last_seen[idx] = current_time

    cv2.putText(
        frame,
        f"Unique people: {total_people}",
        (20, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 0, 255),
        2
    )

    cv2.imshow("video", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

video.release()
cv2.destroyAllWindows()

In [3]:
# شمارش هر فرد فقط یکبار (Unique Person Counting)

import cv2
import threading
import math
import time

# خواندن نام کلاس‌های دیتاست COCO
with open("./data/object_detection_classes_coco.txt", "r") as f:
    class_names = f.read().strip().split("\n")

# بارگذاری مدل SSD MobileNet برای تشخیص اشیا
model = cv2.dnn.readNetFromTensorflow(
    "./data/frozen_inference_graph.pb",
    "./data/ssd_mobilenet_v2_coco_2018_03_29.pbtxt"
)

# اتصال به استریم ویدیو (مثلاً دوربین IP)
video = cv2.VideoCapture("http://192.168.1.100:8080/video")

# کاهش بافر برای کاهش تأخیر
video.set(cv2.CAP_PROP_BUFFERSIZE, 1)

# آخرین فریم دریافت‌شده از دوربین
latest_frame = None

# فریم پردازش‌شده برای نمایش
processed_frame = None

# قفل برای جلوگیری از تداخل بین thread ها
lock = threading.Lock()

# لیست افراد شناسایی‌شده (مرکز هر فرد)A
tracked_people = []  # (cx, cy)

# آستانه فاصله برای تشخیص یک فرد تکراری
DIST_THRESHOLD = 60


# محاسبه فاصله بین دو نقطه
def distance(a, b):
    return math.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2)


# -------------------------
# Thread 1: دریافت فریم از دوربین
# -------------------------
def capture():
    global latest_frame

    while True:
        ret, frame = video.read()

        # اگر فریم دریافت نشد، خروج از حلقه
        if not ret:
            break

        # تغییر اندازه فریم برای سرعت بیشتر پردازش
        frame = cv2.resize(frame, (640, 480))

        # ذخیره امن آخرین فریم
        with lock:
            latest_frame = frame


# -------------------------
# Thread 2: پردازش و تشخیص افراد
# -------------------------
def detect():
    global latest_frame, processed_frame, tracked_people

    while True:

        # اگر هنوز فریمی دریافت نشده
        if latest_frame is None:
            time.sleep(0.01)
            continue

        # کپی امن از فریم برای پردازش
        with lock:
            frame = latest_frame.copy()

        h, w = frame.shape[:2]

        # تبدیل تصویر به blob برای مدل DNN
        blob = cv2.dnn.blobFromImage(frame, 0.007843, (300, 300), 127.5)

        # ارسال ورودی به مدل
        model.setInput(blob)
        output = model.forward()

        boxes = []
        confidences = []

        # -------------------------
        # استخراج نتایج مدل
        # -------------------------
        for det in output[0, 0]:
            score = float(det[2])   # میزان اطمینان
            class_id = int(det[1])  # شناسه کلاس

            # فقط اگر دقت بالا باشد و کلاس معتبر باشد
            if score > 0.4 and class_id <= len(class_names):

                # فقط کلاس "person"
                if class_names[class_id - 1] == "person":

                    # تبدیل مختصات نرمال‌شده به پیکسلی
                    x1 = int(det[3] * w)
                    y1 = int(det[4] * h)
                    x2 = int(det[5] * w)
                    y2 = int(det[6] * h)

                    # ذخیره باکس تشخیص داده شده
                    boxes.append([x1, y1, x2-x1, y2-y1])
                    confidences.append(score)

        # -------------------------
        # حذف تشخیص‌های تکراری (Non-Maximum Suppression)
        # -------------------------
        indices = cv2.dnn.NMSBoxes(boxes, confidences, 0.5, 0.4)

        # اگر چیزی تشخیص داده شد
        if len(indices) > 0:
            for i in indices.flatten():

                x, y, bw, bh = boxes[i]

                # محاسبه مرکز هر فرد
                cx = x + bw // 2
                cy = y + bh // 2

                # بررسی اینکه آیا قبلاً دیده شده یا نه
                already_seen = False

                for p in tracked_people:
                    if distance((cx, cy), p) < DIST_THRESHOLD:
                        already_seen = True
                        break

                # اگر فرد جدید بود، اضافه کن
                if not already_seen:
                    tracked_people.append((cx, cy))

                # رسم باکس روی تصویر
                cv2.rectangle(frame, (x, y), (x+bw, y+bh), (0, 255, 0), 2)

                # رسم نقطه مرکز فرد
                cv2.circle(frame, (cx, cy), 3, (0, 0, 255), -1)

        # نمایش تعداد افراد یکتا
        cv2.putText(frame,
                    f"Unique People: {len(tracked_people)}",
                    (20, 30),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    (0, 0, 255),
                    2)

        # ذخیره فریم پردازش‌شده برای نمایش
        processed_frame = frame

        # جلوگیری از مصرف CPU زیاد
        time.sleep(0.01)


# -------------------------
# اجرای Thread ها
# -------------------------
t1 = threading.Thread(target=capture)
t2 = threading.Thread(target=detect)

t1.start()
t2.start()

# -------------------------
# نمایش ویدیو در حلقه اصلی
# -------------------------
while True:
    if processed_frame is not None:
        cv2.imshow("People Counter", processed_frame)

    # خروج با کلید q
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# آزادسازی منابع
video.release()
cv2.destroyAllWindows()